# Exp2：基于回归分析的大学综合得分预测
---

## 一、案例简介
大学排名是一个非常重要同时也极富挑战性与争议性的问题，一所大学的综合实力涉及科研、师资、学生等方方面面。目前全球有上百家评估机构会评估大学的综合得分进行排序，而这些机构的打分也往往并不一致。在这些评分机构中，世界大学排名中心（Center for World University Rankings，缩写CWUR）以评估教育质量、校友就业、研究成果和引用，而非依赖于调查和大学所提交的数据著称，是非常有影响力的一个。

本任务中我们将根据 CWUR 所提供的世界各地知名大学各方面的排名（师资、科研等），一方面通过数据可视化的方式观察不同大学的特点，另一方面希望构建机器学习模型（线性回归）预测一所大学的综合得分。

## 二、作业说明
使用来自 Kaggle 的[数据](https://www.kaggle.com/mylesoneill/world-university-rankings?select=cwurData.csv)，构建「线性回归」模型，根据大学各项指标的排名预测综合得分。

**基本要求：**
* 按照 8:2 随机划分训练集测试集，用 RMSE 作为评价指标，得到测试集上线性回归模型的 RMSE 值；
* 对线性回归模型的系数进行分析。

**扩展要求：**
* 对数据进行观察与可视化，展示数据特点；
* 尝试其他的回归模型，对比效果；
* 尝试将离散的地区特征融入线性回归模型，并对结果进行对比。

**注意事项：**
* 基本输入特征有 8 个：`quality_of_education`, `alumni_employment`, `quality_of_faculty`, `publications`, `influence`, `citations`, `broad_impact`, `patents`；
* 预测目标为`score`；
* 可以使用 sklearn 等第三方库，不要求自己实现线性回归；
* 需要保留所有数据集生成、模型训练测试的代码；

## 三、数据概览

假设数据文件位于当前文件夹，我们用 pandas 读入标准 csv 格式文件的函数`read_csv()`将数据转换为`DataFrame`的形式。观察前几条数据记录：

In [ ]:
import pandas as pd
import numpy as np

data_df = pd.read_csv("./cwurData.csv")  # 读入 csv 文件为 pandas 的 DataFrame
data_df.head(3).T  # 观察前几列并转置方便观察

去除其中包含 NaN 的数据，保留 2000 条有效记录。

In [ ]:
data_df = data_df.dropna()  # 舍去包含NaN的row
len(data_df)

取出对应自变量以及因变量的列，之后就可以基于此切分训练集和测试集，并进行模型构建与分析。

In [ ]:
feature_cols = [
    "quality_of_faculty",
    "publications",
    "citations",
    "alumni_employment",
    "influence",
    "quality_of_education",
    "broad_impact",
    "patents",
]
X = data_df[feature_cols]
Y = data_df["score"]
X

## 四、模型构建

### 导入相关库

In [ ]:
import seaborn as sns  # 导入画图库
import matplotlib.pyplot as plt  # 导入画图库
from sympy import Matrix  # 导入SymPy矩阵库，用于符号矩阵运算和简化
from sklearn.pipeline import make_pipeline  # 导入机器学习流包
from sklearn.preprocessing import StandardScaler  # 导入标准化包和独热编码包
from sklearn.linear_model import LinearRegression  # 导入线性回归模型
from sklearn.model_selection import train_test_split  # 导入训练集和测试机划分模块
from sklearn.tree import DecisionTreeRegressor  # 导入决策树回归模型
from sklearn.ensemble import RandomForestRegressor  # 导入随机森林回归模型

### 检查特征之间是否存在多重共线性

In [ ]:
A = X.to_numpy()
matrix_A = Matrix(A)
rref_matrix, pivot_cols = matrix_A.rref()
print(int(np.linalg.matrix_rank(A)))
print(len(list(pivot_cols)))

**特征之间不存在多重共线性**

### 定义均方根误差计算函数

In [ ]:
def RMSE(y_hat, y_true):
    """
    计算均方根误差（RMSE）

    参数:
        y_hat (np.ndarray): 模型预测值（形状需与y_true一致）
        y_true (np.ndarray): 真实值（标签）

    返回:
        float: RMSE值（非负）

    示例:
        >>> y_true = np.array([1, 2, 3])
        >>> y_hat = np.array([1.2, 1.8, 3.1])
        >>> RMSE(y_hat, y_true)
        0.2160246899469287
    """

    # 输入检查：确保形状一致
    if y_hat.shape != y_true.shape:
        raise ValueError("预测值和标签形状不一致")

    # 计算RMSE
    mse = np.mean(np.square(y_true - y_hat))  # 均方误差

    return float(np.sqrt(mse))

### 划分训练集与测试集

In [ ]:
RANDOM_STATE = 2025
X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.2, random_state=RANDOM_STATE
)

### 未对特征标准化后的线性回归

In [ ]:
# 模型实例化
LR = LinearRegression()  # 线性回归模型

# 训练模型
LR.fit(X_train, y_train)

# 在测试集上进行预测
y_predict = LR.predict(X_test)

# 计算均方根误差
RMSE(y_predict, y_test)

### 对特征进行标准化的线性回归

In [ ]:
# 模型实例化
pipeline = make_pipeline(
    StandardScaler(),  # 数据标准化
    LinearRegression(),  # 线性回归模型
)

# 训练模型
pipeline.fit(X_train, y_train)

# 在测试集上进行预测
y_pred = pipeline.predict(X_test)

# 计算均方根误差
RMSE(y_pred, y_test)

In [ ]:
# 获取相应的评估器
first_estimator = pipeline.steps[0][1]
second_estimator = pipeline.steps[1][1]

In [ ]:
# 查看系数参数
second_estimator.coef_

In [ ]:
# 查看截距参数
second_estimator.intercept_

**可见特征是否进行标准化对于线性回归的预测结果几乎没有影响**

&emsp;&emsp;从线性回归在训练集上的结果看，各相参数对应的权重为：

|特征|特征名称|系数|
|-|-|-|
|'quality_of_faculty'|**师资质量**|-3.39789913|
|'publications'|**出版物数量**|0.04742167|
|'citations'|**被引用次数**|-0.04946043|
|'alumni_employment'|**校友就业情况**|-1.27061751|
|'influence'|**影响力**|0.16139674|
|'quality_of_education'|**教育质量**|-0.6242429|
|'broad_impact'|**广泛影响力**|-0.64690301|
|'patents'|**专利数量**|-0.61007473|

&emsp;&emsp;从上面的表格可以看出：
* 学校的**师资质量**、**被引用次数**、**校友就业情况**、**教育质量**、**广泛影响力**、**专利数量**的排名与学校最终得分是负相关的。
* 学校**出版物数量**、**影响力**的排名与学校的最终得分是正相关的。
* 从系数的绝对值来看，从大到小依次是：**师资质量**、**校友就业情况**、**广泛影响力**、**教育质量**、**专利数量**、**被引用次数**、**出版物数量**、**影响力**。也是一个学校这些方面的排名对最终得分的影响大小的排序。很明显**师资质量**对学校的最终得分影响最大，**影响力**对学校的最终得分影响最小。

**总结：**
* 按照经验，所有的特征应该与学校的最终得分都应该是负相关的，也就是说，线性回归得到的系数应该全都是负的，学校在任何一个方面的排名提高，都应该能够提高学校的得分，这说明线性回归模型并不能很好的学习到数据中的模式。
* 从预测结果来看，预测结果的均方根误差约为<font color="red">**4.05**</font>，同样也能够在一定程度上说明线性回归模型并不能很好的学习到数据中的模式。

### 拓展部分

#### 要求一：对数据进行观察与可视化，展示数据特点

**展示不同国家的高校数量**

In [ ]:
region_counts = data_df["region"].value_counts().sort_values(ascending=False)
sorted_regions = region_counts.index  # 获取排序后的地区列表

plt.figure(figsize=(12, 7))
sns.countplot(
    x="region",
    data=data_df,
    order=sorted_regions,  # 按数量降序排列地区
    hue="region",  # 指定hue为当前变量以应用调色板
    palette="viridis",  # 使用渐变颜色调色板
    dodge=False,  # 关闭分组间距
)

plt.legend([], [], frameon=False)

plt.title("Regional Distribution of Samples", fontsize=14, pad=20)
plt.xlabel("Region", fontsize=16, color="purple")
plt.ylabel("Number", fontsize=16, color="purple")
plt.xticks(rotation=90, fontsize=10)
plt.grid(axis="y", linestyle="--", alpha=0.7)

for p in plt.gca().patches:
    height = int(p.get_height())
    plt.gca().annotate(
        f"{height}",
        xy=(p.get_x() + p.get_width() / 2, height),
        xytext=(0, 3),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=6,
    )

plt.tight_layout()
plt.show()

#### 要求二：尝试其他的回归模型，对比效果

**决策树回归模型**

In [ ]:
# 构建机器学习流，包括标准化和逻辑回归模型
pipeline = make_pipeline(
    StandardScaler(),  # 数据标准化
    DecisionTreeRegressor(random_state=RANDOM_STATE),  # 决策树回归模型
)

# 训练模型
pipeline.fit(X_train, y_train)

# 在测试集上进行预测
y_pred = pipeline.predict(X_test)

RMSE(y_pred, y_test)

**随机森林回归模型**

In [ ]:
# 构建机器学习流，包括标准化和逻辑回归模型
pipeline = make_pipeline(
    StandardScaler(),  # 数据标准化
    RandomForestRegressor(random_state=RANDOM_STATE),  # 随机森林回归模型
)

# 训练模型
pipeline.fit(X_train, y_train)

# 在测试集上进行预测
y_pred = pipeline.predict(X_test)

RMSE(y_pred, y_test)

**三种机器学习回归模型的对比如下：**
<table style="
  border-collapse: collapse; 
  width: 50%; 
  font-family: Arial, sans-serif; 
  margin: 20px auto;
">
  <thead>
    <tr>
      <th style="
        padding: 15px; 
        text-align: center; 
        background-color: #e0f2f1; 
        color: #1a73e8; 
        font-weight: bold; 
        border: 1px solid #ddd;
        border-right: 2px solid #ddd;
      ">回归模型</th>
      <th style="
        padding: 15px; 
        text-align: center; 
        background-color: #e0f2f1; 
        color: #1a73e8; 
        font-weight: bold; 
        border: 1px solid #ddd;
      ">均方根误差</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="
        padding: 12px; 
        text-align: center; 
        border: 1px solid #ddd; 
        border-right: 2px solid #ddd;
        color: #5f6368;
      "><b>线性回归</b></td>
      <td style="
        padding: 12px; 
        text-align: center; 
        border: 1px solid #ddd; 
        color: #d32f2f;
        font-weight: bold;
      ">4.051009</td>
    </tr>
    <tr>
      <td style="
        padding: 12px; 
        text-align: center; 
        border: 1px solid #ddd; 
        border-right: 2px solid #ddd;
        color: #5f6368;
      "><b>决策树回归</b></td>
      <td style="
        padding: 12px; 
        text-align: center; 
        border: 1px solid #ddd; 
        color: #d32f2f;
        font-weight: bold;
      ">0.847164</td>
    </tr>
    <tr>
      <td style="
        padding: 12px; 
        text-align: center; 
        border: 1px solid #ddd; 
        border-right: 2px solid #ddd;
        color: #5f6368;
      "><b>随机森林回归</b></td>
      <td style="
        padding: 12px; 
        text-align: center; 
        border: 1px solid #ddd; 
        color: #d32f2f;
        font-weight: bold;
      ">0.539878</td>
    </tr>
  </tbody>
</table>

**可见，模型在测试集上的标表现：** <font color="purple">**随机森林回归 > 决策树回归 > 线性回归**。<font>

#### 要求三：尝试将离散的地区特征融入线性回归模型，并对结果进行对比

In [ ]:
# 重新读入数据
data = pd.read_csv("./cwurData.csv")
# 舍去包含NaN的行
data = data.dropna()

In [ ]:
# 划分特征和标签，特征包含地区特征
feature_cols = [
    "region",
    "quality_of_faculty",
    "publications",
    "citations",
    "alumni_employment",
    "influence",
    "quality_of_education",
    "broad_impact",
    "patents",
]
X = data[feature_cols]
Y = data["score"]
X.head(5)

In [ ]:
# 对离散型变量'region'进行one_hot_encoding
X = pd.get_dummies(X, columns=["region"], drop_first=False)
X.head(5)

**独热编码后总共有67个特征**

In [ ]:
# 检查特征之间是否存在多重共线性
A = X.to_numpy().astype(float)  # 布尔值True会转为1，False会转为0
matrix_A = Matrix(A)
rref_matrix, pivot_cols = matrix_A.rref()
print(int(np.linalg.matrix_rank(A)))
print(len(list(pivot_cols)))

**特征之间不存在多重共线性**

In [ ]:
# 进行训练集与测试集的划分
RANDOM_STATE = 2025
X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.2, random_state=RANDOM_STATE
)

In [ ]:
# 对特征标准化后的线性回归
# 模型实例化
pipeline = make_pipeline(
    StandardScaler(),  # 数据标准化
    LinearRegression(),  # 线性回归模型
)

# 训练模型
pipeline.fit(X_train, y_train)

# 在测试集上进行预测
y_pred = pipeline.predict(X_test)

# 计算均方根误差
RMSE(y_pred, y_test)

In [ ]:
# 未对特征进行标准化的线性回归
# 模型实例化
LR = LinearRegression()  # 线性回归模型

# 训练模型
LR.fit(X_train, y_train)

# 在测试集上进行预测
y_predict = LR.predict(X_test)

# 计算均方根误差
RMSE(y_predict, y_test)

<font color="purple">**对特征标准化后的线性回归的模型效果优于未对特征进行标准化的线性回归的模型效果。**</font>

**将离散的地区特征融入线性回归模型后的结果对比**
| 特征处理方式             | 均方根误差       |
|--------------------------|------------------|
| 未融入离散的地区特征     | **4.051009**     |
| 融入离散的地区特征       | **3.991466**     |

**地区特征处理对模型的影响：** <font color="red">**融入离散地区特征后的线性回归模型表现优于未处理离散地区特征特征的模型**。</font>